# Running the Classifier



## Setting up the classes

The example data is stored in the repository with Git LFS, so the notebook needs a clone of the repository and not only the installed package.

Run the next cell in Google Colab or any other fresh environment. Skip it when the notebook was opened from a checkout of the repository in which `git lfs pull` has already been run.

In [ ]:
!git clone https://github.com/BRAINSia/dcm-classifier.git
!git -C dcm-classifier lfs install --local
!git -C dcm-classifier lfs pull

In [ ]:
%pip install dcm-classifier

In [ ]:
from pathlib import Path

from dcm_classifier.image_type_inference import ImageTypeClassifierBase
from dcm_classifier.study_processing import ProcessOneDicomStudyToVolumesMappingBase

# The repository is the checkout this notebook was opened from, or the clone made above.
repository_directory: Path | None = next(
    (
        candidate
        for candidate in [
            Path.cwd(),
            *Path.cwd().parents,
            Path.cwd() / "dcm-classifier",
        ]
        if (candidate / "tests" / "testing_data").is_dir()
    ),
    None,
)
if repository_directory is None:
    msg = "The dcm-classifier repository was not found. Run the clone cell above first."
    raise FileNotFoundError(msg)

session_directory = (
    repository_directory
    / "tests"
    / "testing_data"
    / "anonymized_testing_data"
    / "anonymized_data"
)

# create inferer object
inferer = ImageTypeClassifierBase()

# create study for given session directory
study = ProcessOneDicomStudyToVolumesMappingBase(
    study_directory=session_directory.as_posix(), inferer=inferer
)

# run the inference on the study
study.run_inference()

## DICOM Selection

In many applications we are interested in only specific modalities or images acquired along specific plane. dcm-classifier makes it easy to make this selection. 

The code below shows how to parse all images within the study and how to choose a specific image, in this example it: Axial T1w

NOTE: we can parse on a series or volume level. In most cases series has only one volume but in some cases such as Diffusion Weighted Imaging a series could have multiple volumes and have different `.get_volume_modality()` and `.get_series_modality()` outputs (Example: Volume 'b0' Series: 'dwig' meaning the B0 volume is one of the components of DWI image)

In [ ]:
ax_t1w = None
for series_number, series in study.series_dictionary.items():
    for volume in series.get_volume_list():
        volume_modality = volume.get_volume_modality()
        volume_acq_plane = volume.get_acquisition_plane()
        print(
            f"Series Number: {series_number} | Modality: {volume_modality} | Acq Plane: {volume_acq_plane}"
        )
        if volume_modality == "t1w" and volume_acq_plane == "ax":
            ax_t1w = volume.get_itk_image()

Our volume of interest is the T1w Axial image returned as ITK image

In [ ]:
print(type(ax_t1w))
print(ax_t1w)

## Saving Sequences

Saving volumetric data can be done in two ways here - using the ITK library or the popular [dcm2niix](https://github.com/rordenlab/dcm2niix) package. For these examples, we are saving the Axial T1W selected earlier. 

Note: the example data used has been anonymized and the pixel intensity information was set as 0 for all voxels

#### Using ITK

Once the image is obtained, the image can easily be saved using ITK. ITK is the preferred method for saving the images as it supports all image type supported by ITK such as NIfTI and NRRD in the example below.

In [ ]:
import itk

# let's follow the BIDS naming system
# Note: the information about the study or patient can be obtained using pydicom and a .dcm file corresponding to the volume
# However, here we will make up the subject and session ids
sub = "sub-01"
ses = "ses-01"
nifti_name = f"{sub}_{ses}_desc-ax_t1w.nii.gz"
nrrd_name = f"{sub}_{ses}_desc-ax_t1w.nrrd"

# save image as nifti and nrrd
itk.imwrite(ax_t1w, nifti_name)
itk.imwrite(ax_t1w, nrrd_name)

#### Handling multiple images of the same modality

We can have more than one image of the same modality. Here is a simple way how to save all the data utilizing the `run` parameter of BIDS format.

In [ ]:
sub = "sub-01"
ses = "ses-01"
output_dir = "."
suffix = ".nii.gz"
desc = "usingITK"

for series in study.series_dictionary.values():
    for volume in series.get_volume_list():
        modality = volume.get_volume_modality()
        plane = volume.get_acquisition_plane()
        if modality == "t1w" and plane == "ax":
            itk_volume = volume.get_itk_image()  # get image
            run_num = 1
            fname = f"{sub}_{ses}_run-00{run_num}_acq-{plane}_desc-{desc}_{modality}{suffix}"  # set filename
            while Path(
                f"{output_dir}/{fname}"
            ).exists():  # ensure we increase run number instead of overwriting
                fname = fname.replace(f"run-00{run_num}", f"run-00{run_num + 1}")
                run_num += 1
            itk.imwrite(itk_volume, fname)

#### Using dcm2niix

Using dcm2niix is more complicated. We provide a function that takes path to the dcm2niix script together with other information.

In [ ]:
from subprocess import run


def create_nifti_file(
    dcm2niix_path: str,
    subject: str,
    session: str,
    modality: str,
    plane: str,
    output_dir: str,
    dcm_dir: str,
    desc: str | None = None,
) -> None:
    run_num = 1
    if desc is not None:
        fname = (
            f"{subject}_{session}_run-00{run_num}_acq-{plane}_desc-{desc}_{modality}"
        )
    else:
        fname = f"{subject}_{session}_run-00{run_num}_acq-{plane}_{modality}"
    while Path(f"{output_dir}/{fname}.nii.gz").exists():
        fname = fname.replace(f"run-00{run_num}", f"run-00{run_num + 1}")
        run_num += 1
    run([dcm2niix_path, "-o", output_dir, "-f", fname, "-z", "y", dcm_dir], check=False)
    print(f"{dcm2niix_path} -o {output_dir} -f {fname} -n -z y {dcm_dir}")

For robustness, the code below will move all individual DICOM files corresponding to the volume to a temporary directory. Then the dcm2niix script uses that directory to read the files and save the output. The advantage of using the dcm2niix is that it produces the corresponding .json file the process is much more complicated and and slower comparing to ITK option above. 

In [ ]:
import shutil
import tempfile

# dcm2niix is found on the PATH; set the path by hand if it is installed elsewhere
dcm2niix_path = shutil.which("dcm2niix")

sub = "sub-01-test"
ses = "ses-01-test"
desc = "usingdcm2niix"

final_dir_path = "."

if dcm2niix_path is None:
    print("dcm2niix was not found, skipping this example")
else:
    for series in study.series_dictionary.values():
        for volume in series.get_volume_list():
            if (
                volume.get_volume_modality() == "t1w"
                and volume.get_acquisition_plane() == "ax"
            ):
                temp_dir = tempfile.mkdtemp()
                file_list = volume.get_one_volume_dcm_filenames()
                for file in file_list:
                    shutil.copy(file, temp_dir)
                create_nifti_file(
                    dcm2niix_path=dcm2niix_path,
                    subject=sub,
                    session=ses,
                    modality=volume.get_volume_modality(),
                    plane=volume.get_acquisition_plane(),
                    output_dir=final_dir_path,
                    dcm_dir=temp_dir,
                    desc=desc,
                )